## Usefull libraries:

In [617]:

import numpy as np
from scipy.stats import chi2
import glob
import camb
from camb import model, initialpower
import math
import matplotlib.pyplot as plt
from scipy import special
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.integrate import simpson
from scipy.integrate import trapezoid
from CosmoFunc import *
from BF_OptMC import *
from lu_functions import *
import time
import os
from numba import jit, prange

## Calculation of the mean R matrix from the 10 samples

In [618]:

# paramètres:
N = 3000 # nombre de galaxies qu'on prend


# Pattern pour capturer tous les fichiers avec ce format
fichiers = sorted(glob.glob(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_{N}g_sample_*.txt'))

# Extraire toutes les matrices
R = []
for fichier in fichiers:
    mat = np.loadtxt(fichier, skiprows=2)
    R.append(mat)

# Stocker dans des variables R1, R2, etc.
for i, mat in enumerate(R, 1):
    globals()[f'R{i}'] = mat
    
# Calculer la matrice moyenne (covariance moyenne)
R_mean = np.mean(R, axis=0)

# Vous pouvez aussi calculer l'écart-type entre les runs
R_std = np.std(R, axis=0)

# Erreur standard de la moyenne (incertitude sur la moyenne)
R_sem = R_std / np.sqrt(len(R))  # 10 runs


# R_thorie est la resulats de Fei:

R_theorie = np.array([
    [5248.882, -58.604, -2725.658, -1.093, 1.034, -9.032, -0.357, 0.153, 12.569],
    [-58.604, 7048.148, 1998.970, 3.336, 0.974, 0.819, 1.109, -13.604, -9.018],
    [-2725.658, 1998.970, 45434.716, 30.027, 3.378, 18.922, 17.950, -0.134, -203.080],
    [-1.093, 3.336, 30.027, 0.140, 0.005, 0.012, 0.061, -0.005, -0.093],
    [1.034, 0.974, 3.378, 0.005, 0.036, -0.002, 0.005, 0.003, -0.016],
    [-9.032, 0.819, 18.922, 0.012, -0.002, 0.085, 0.014, 0.003, -0.085],
    [-0.357, 1.109, 17.950, 0.061, 0.005, 0.014, 0.152, 0.003, -0.042],
    [0.153, -13.604, -0.134, -0.005, 0.003, 0.003, 0.003, 0.098, -0.003],
    [12.569, -9.018, -203.080, -0.093, -0.016, -0.085, -0.042, 0.003, 1.103]])

# Calculer la différence normalisée par l'erreur
difference_normalisee = (R_theorie - R_mean) / R_sem

with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt', 'w') as f:
    f.write(f"R matrix mean with 10 samples with {N} galaxies each :\n\n")
    for row in R_mean:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
    f.write(f"R matrix ecart-type : \n\n")
    for row in R_std:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write(f"R matrix standard error on the mean :\n\n")        
    for row in R_sem:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")  
    f.write("(R_Fei - R_mean) / R_error :\n\n")
    for row in difference_normalisee:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")       

print(f"Results saved in: {f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_{N}.txt'}")


with open(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt', 'w') as f:
    f.write(f"R matrix exact from Fei Qin :\n\n")
    for row in R_theorie:
        f.write("  ".join(f"{x:10.4f}" for x in row) + "\n")
        
print(f"Results saved in:{f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt'}")


Results saved in: /renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_random_moy_3000.txt
Results saved in:/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_exact_Fei.txt


## Extraire la prédiction théorique sur le Bulk flow

In [627]:
# function that gives the CRMS for the bulk flow and shear moments from the R matrix prediction from LambdaCDM:

def load_theoretical_CRMS(R):

    # Bulk Flow CRMS
    B_pq = R[:3, :3]
    Bx_CRMS = np.sqrt(B_pq[0, 0])
    By_CRMS = np.sqrt(B_pq[1, 1])
    Bz_CRMS = np.sqrt(B_pq[2, 2])
    sigma_B = np.sqrt(np.trace(B_pq))
    Bp_CRMS = np.sqrt(2/3) * sigma_B
    
    # Shear moments CRMS
    R_shear = R[3:8, 3:8]
    CRMS_Qxx = np.sqrt(R_shear[0, 0])
    CRMS_Qxy = np.sqrt(R_shear[1, 1])
    CRMS_Qxz = np.sqrt(R_shear[2, 2])
    CRMS_Qyy = np.sqrt(R_shear[3, 3])
    CRMS_Qyz = np.sqrt(R_shear[4, 4])
    CRMS_Qzz = np.sqrt(np.abs(-R_shear[0, 0] - R_shear[3, 3]))
    
    return Bx_CRMS,By_CRMS, Bz_CRMS, Bp_CRMS, sigma_B, CRMS_Qxx, CRMS_Qxy,CRMS_Qxz, CRMS_Qyy,CRMS_Qyz, CRMS_Qzz
    

In [ ]:
# for the mean matrix we caluated higher 
Bx_CRMS, By_CRMS, Bz_CRMS, B_p_CRMS, sigma_B, CRMS_Qxx, CRMS_Qxy, CRMS_Qxz, CRMS_Qyy, CRMS_Qyz, CRMS_Qzz = load_theoretical_CRMS(R_mean)

print('The theoritical Bulk Flow Bx = 0 +-',Bx_CRMS,'km/s')
print('The theoritical Bulk Flow By = 0 +-',By_CRMS,'km/s')
print('The theoritical Bulk Flow Bz = 0 +-',Bz_CRMS,'km/s')
print('The theoritical Bulk Flow amplitude Bp =' ,B_p_CRMS, '+-',sigma_B,'km/s')

print("Prédiction ΛCDM pour les shear moments :")
print(f"Qxx = 0 ± {CRMS_Qxx:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qxy = 0 ± {CRMS_Qxy:.4f} km/s")
print(f"Qxz = 0 ± {CRMS_Qxz:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy:.4f} km/s")
print(f"Qyz = 0 ± {CRMS_Qyz:.4f} km/s")
print(f"Qzz = 0 ± {CRMS_Qzz:.4f} km/s")

The theoritical Bulk Flow Bx = 0 +- 73.23988974595743 km/s
The theoritical Bulk Flow By = 0 +- 86.18153711787694 km/s
The theoritical Bulk Flow Bz = 0 +- 217.83607412455817 km/s
The theoritical Bulk Flow amplitude Bp = 200.40607771888224 +- 245.44631588190524 km/s
Prédiction ΛCDM pour les shear moments :
Qxx = 0 ± 0.3716 km/s
Qyy = 0 ± 0.3941 km/s
Qxy = 0 ± 0.1920 km/s
Qxz = 0 ± 0.3052 km/s
Qyy = 0 ± 0.3941 km/s
Qyz = 0 ± 0.3242 km/s
Qzz = 0 ± 0.5417 km/s


In [ ]:
# for fei R_pq matrix 
Bx_CRMS_fei, By_CRMS_fei, Bz_CRMS_fei, B_p_CRMS_fei, sigma_B_fei, CRMS_Qxx_fei, CRMS_Qxy_fei, CRMS_Qxz_fei, CRMS_Qyy_fei, CRMS_Qyz_fei, CRMS_Qzz_fei = load_theoretical_CRMS(R_theorie)

print('The theoritical Bulk Flow Bx with Fei matrix = 0 +-',Bx_CRMS_fei,'km/s')
print('The theoritical Bulk Flow By with Fei matrix = 0 +-',By_CRMS_fei,'km/s')
print('The theoritical Bulk Flow Bz with Fei matrix = 0 +-',Bz_CRMS_fei,'km/s')
print('The theoritical Bulk Flow amplitude with Fei matrix =', B_p_CRMS_fei, '+-',sigma_B_fei,'km/s')

print("Prédiction ΛCDM pour les shear moments :")
print(f"Qxx = 0 ± {CRMS_Qxx_fei:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy_fei:.4f} km/s")
print(f"Qxy = 0 ± {CRMS_Qxy_fei:.4f} km/s")
print(f"Qxz = 0 ± {CRMS_Qxz_fei:.4f} km/s")
print(f"Qyy = 0 ± {CRMS_Qyy_fei:.4f} km/s")
print(f"Qyz = 0 ± {CRMS_Qyz_fei:.4f} km/s")
print(f"Qzz = 0 ± {CRMS_Qzz_fei:.4f} km/s")

The theoritical Bulk Flow Bx with Fei matrix = 0 +- 72.44916838722168 km/s
The theoritical Bulk Flow By with Fei matrix = 0 +- 83.95324889484623 km/s
The theoritical Bulk Flow Bz with Fei matrix = 0 +- 213.1542070896092 km/s
The theoritical Bulk Flow amplitude with Fei matrix = 196.18315591983597 +- 240.27431406623555 km/s
Prédiction ΛCDM pour les shear moments :
Qxx = 0 ± 0.3742 km/s
Qyy = 0 ± 0.3899 km/s
Qxy = 0 ± 0.1897 km/s
Qxz = 0 ± 0.2915 km/s
Qyy = 0 ± 0.3899 km/s
Qyz = 0 ± 0.3130 km/s
Qzz = 0 ± 0.5404 km/s


## extraire l'amplitude du bulk flow de nos estimateurs

In [653]:
def load_data_results(filepath_bulk, filepath_shear,file_path_mocks_B, file_path_Q):

    # Chargement Bulk Flow
    Bx, By, Bz = np.loadtxt(filepath_bulk, skiprows=6, usecols=1, max_rows=3)
    err_Bx, err_By, err_Bz = np.loadtxt(filepath_bulk, skiprows=6, usecols=2, max_rows=3)
    
    
    # Calcul amplitude Bulk Flow
    B_ampl = np.sqrt(Bx**2 + By**2 + Bz**2)
    err_B_ampl = (np.abs(Bx*err_Bx) + np.abs(By*err_By) + np.abs(Bz*err_Bz)) / B_ampl
    
    # Chargement Shear moments
    Qxx = np.loadtxt(filepath_shear, skiprows=8,usecols=1,max_rows=1)
    Qxy = np.loadtxt(filepath_shear, skiprows=9, usecols=1, max_rows=1)
    Qxz = np.loadtxt(filepath_shear, skiprows=10, usecols=1, max_rows=1)
    Qyy = np.loadtxt(filepath_shear, skiprows=11, usecols=1, max_rows=1)
    Qyz = np.loadtxt(filepath_shear, skiprows=12, usecols=1, max_rows=1)
    Qzz = np.loadtxt(filepath_shear, skiprows=13, usecols=1, max_rows=1)
    
    # Erreurs Shear
    err_Qxx = np.loadtxt(filepath_shear, skiprows=8, usecols=2, max_rows=1)
    err_Qxy = np.loadtxt(filepath_shear, skiprows=9, usecols=2, max_rows=1)
    err_Qxz = np.loadtxt(filepath_shear, skiprows=10, usecols=2, max_rows=1)
    err_Qyy = np.loadtxt(filepath_shear, skiprows=11, usecols=2, max_rows=1)
    err_Qyz = np.loadtxt(filepath_shear, skiprows=12, usecols=2, max_rows=1)
    err_Qzz = np.loadtxt(filepath_shear, skiprows=13, usecols=2, max_rows=1)
    
    # Chargement matrice de covariance Shear (lignes 17-21 du fichier shear)
    cov = np.loadtxt(filepath_shear, skiprows=16, max_rows=8)
    
    # chargement des mocks:
    all_mocks_B = np.loadtxt(file_path_mocks_B, skiprows=1)
    all_mocks_Q = np.loadtxt(file_path_Q, skiprows=1)

    Bx_mocks_CRMS = np.std(all_mocks_B[:, 1] , ddof=1)
    By_mocks_CRMS = np.std(all_mocks_B[:, 2] , ddof=1)
    Bz_mocks_CRMS = np.std(all_mocks_B[:, 3] , ddof=1)
    Ux_mocks_CRMS = np.std(all_mocks_B[:, 4] , ddof=1)
    Uy_mocks_CRMS = np.std(all_mocks_B[:, 5] , ddof=1)
    Uz_mocks_CRMS = np.std(all_mocks_B[:, 6] , ddof=1)
    
    # we extract the values of Q and the true values of for all the mocks:
    Qxx_opt_mocks = np.std(all_mocks_Q[:, 1])
    Qxy_opt_mocks = np.std(all_mocks_Q[:, 2])  
    Qxz_opt_mocks = np.std(all_mocks_Q[:, 3])
    Qyy_opt_mocks = np.std(all_mocks_Q[:, 4])  
    Qyz_opt_mocks = np.std(all_mocks_Q[:, 5])  
    Qzz_opt_mocks = np.std(all_mocks_Q[:, 6]) 

    Qxx_true_mocks = np.std(all_mocks_Q[:,6])
    Qxy_true_mocks = np.std(all_mocks_Q[:,7])
    Qxz_true_mocks = np.std(all_mocks_Q[:,8])
    Qyy_true_mocks = np.std(all_mocks_Q[:,9])
    Qyz_true_mocks = np.std(all_mocks_Q[:,10])
    Qzz_true_mocks = np.std(all_mocks_Q[:,11])
    
    return Bx, By, Bz, B_ampl, err_Bx, err_By,  err_Bz, err_B_ampl, Qxx,  Qxy, Qxz, Qyy, Qyz,Qzz ,err_Qxx, err_Qxy, err_Qxz, err_Qyy,  err_Qyz, err_Qzz, cov , Bx_mocks_CRMS, By_mocks_CRMS, Bz_mocks_CRMS, Ux_mocks_CRMS, Uy_mocks_CRMS, Uz_mocks_CRMS, Qxx_opt_mocks, Qxy_opt_mocks, Qxz_opt_mocks, Qyy_opt_mocks, Qyz_opt_mocks, Qzz_opt_mocks,Qxx_true_mocks, Qxy_true_mocks, Qxz_true_mocks, Qyy_true_mocks, Qyz_true_mocks, Qzz_true_mocks

In [654]:
# results of the DESI V5 file:
Input   = '/renoir/fromenti/Documents/codes_Bulk_flow/results/'
Bx_eta_v5, By_eta_v5, Bz_eta_v5, B_ampl_eta_v5, err_Bx_eta_v5, err_By_eta_v5,  err_Bz_eta_v5, err_B_ampl_eta_v5, Qxx_eta_v5,  Qxy_eta_v5, Qxz_eta_v5, Qyy_eta_v5, Qyz_eta_v5,Qzz_eta_v5 ,err_Qxx_eta_v5, err_Qxy_eta_v5, err_Qxz_eta_v5, err_Qyy_eta_v5,  err_Qyz_eta_v5, err_Qzz_eta_v5 , cov_eta_v5 , Bx_mocks_CRMS_eta_v5, By_mocks_CRMS_eta_v5, Bz_mocks_CRMS_eta_v5, Ux_mocks_CRMS_eta_v5, Uy_mocks_CRMS_eta_v5, Uz_mocks_CRMS_eta_v5, Qxx_opt_mocks_eta_v5, Qxy_opt_mocks_eta_v5, Qxz_opt_mocks_eta_v5, Qyy_opt_mocks_eta_v5, Qyz_opt_mocks_eta_v5, Qzz_opt_mocks_eta_v5,Qxx_true_mocks_eta_v5, Qxy_true_mocks_eta_v5, Qxz_true_mocks_eta_v5, Qyy_true_mocks_eta_v5, Qyz_true_mocks_eta_v5, Qzz_true_mocks_eta_v5 = load_data_results(Input + 'Bulk_flow_data_etaMLE.txt', Input + 'shear_moment_data_with_Qzz_etaMLE.txt', Input + 'Bulk_flow_all_mocks_etaMLE.txt', Input + 'Shear_moment_all_mocks_with_Qzz_etaMLE_final.txt')
Bx_w_v5, By_w_v5, Bz_w_v5, B_ampl_w_v5, err_Bx_w_v5, err_By_w_v5,  err_Bz_w_v5, err_B_ampl_w_v5, Qxx_w_v5,  Qxy_w_v5, Qxz_w_v5, Qyy_w_v5, Qyz_w_v5,Qzz_w_v5 ,err_Qxx_w_v5, err_Qxy_w_v5, err_Qxz_w_v5, err_Qyy_w_v5,  err_Qyz_w_v5, err_Qzz_w_v5, cov_w_v5 ,Bx_mocks_CRMS_w_v5, By_mocks_CRMS_w_v5, Bz_mocks_CRMS_w_v5, Ux_mocks_CRMS_w_v5, Uy_mocks_CRMS_w_v5, Uz_mocks_CRMS_w_v5, Qxx_opt_mocks_w_v5, Qxy_opt_mocks_w_v5, Qxz_opt_mocks_w_v5, Qyy_opt_mocks_w_v5, Qyz_opt_mocks_w_v5, Qzz_opt_mocks_w_v5,Qxx_true_mocks_w_v5, Qxy_true_mocks_w_v5, Qxz_true_mocks_w_v5, Qyy_true_mocks_w_v5, Qyz_true_mocks_w_v5, Qzz_true_mocks_w_v5 = load_data_results(Input + 'Bulk_flow_data_wMLE.txt', Input + 'shear_moment_data_with_Qzz_wMLE.txt', Input + 'Bulk_flow_all_mocks_wMLE.txt', Input + 'Shear_moment_all_mocks_with_Qzz_wMLE_final.txt')

cov_eta_v5 = np.array([[550.801573 , -19.569822 , 106.683908   , 0.568013   , 0.201634  , -2.426933   ,0.141744 ,   0.381829 ],  # je l'ai réecrite ici afin de ne pas refaire tourner le bulk_flow_calculator à chaque fois c'est tot_cov cette fois
 [  -19.569822   ,1113.376599 , 233.959214   , 0.022437    ,0.692426   , 0.231701  , 0.869721 ,  -4.197613],
 [ 106.683908    ,233.959214   ,2060.846074  ,  0.218228   , 0.357902  ,  5.564905    ,3.243416 ,   1.797453],
 [  0.568013   , 0.022437   , 0.218228    ,0.028405  ,  0.001211   ,-0.004039  ,-0.018927  , -0.005048],
 [  0.201634   , 0.692426 ,   0.357902   , 0.001211   , 0.022604  ,  0.003421 ,  -0.004261  ,  0.00833 ],
 [ -2.426933  ,  0.231701   , 5.564905  , -0.004039   , 0.003421  ,  0.078359,   -0.000533  , -0.000373],
 [  0.141744   , 0.869721  ,  3.243416  , -0.018927  , -0.004261 ,  -0.000533    ,0.066985 ,   0.010434],
 [  0.381829  , -4.197613   , 1.797453  , -0.005048 ,   0.00833   , -0.000373   , 0.010434  ,  0.073084]])  # size (8,8)


cov_w_v5 = np.array([[548.608632,  -88.072764 ,  81.338432 ,   0.468945 ,   0.19248  ,  -2.375878  ,0.413145 ,   0.671231 ],  
 [  -88.072764, 1191.116932  ,109.878514  ,  0.07592    , 0.38751   ,  0.448049  ,0.565176  , -4.591284],
 [  81.338432  ,109.878514, 2051.236626 ,   0.418965  ,  0.19158   ,  5.889013,   2.54551  ,   2.451711],
 [ 0.468945 ,   0.07592  ,   0.418965  ,  0.028465   , 0.001334  ,   -0.005072,   -0.016805 ,  -0.004273],
 [ 0.19248   ,  0.38751   ,  0.19158  ,   0.001334 ,   0.026265 ,   0.005873,   -0.00774   ,  0.011114 ],
 [ -2.375878 ,   0.448049 ,   5.889013 ,  -0.005072  ,  0.005873  ,  0.080864,  -0.003481  ,  0.003639],
 [  0.413145 ,   0.565176 ,   2.54551   , -0.016805,   -0.00774 ,   -0.003481,  0.065308  ,   0.017202 ],
 [  0.671231  , -4.591284 ,   2.451711  , -0.004273  ,  0.011114 ,   0.003639,   0.017202 ,   0.077877]])  # size (8,8)

/tmp/ipykernel_1798285/3261894244.py:29: UserWarning: loadtxt: input contained no data: "/renoir/fromenti/Documents/codes_Bulk_flow/results/shear_moment_data_with_Qzz_etaMLE.txt"
  cov = np.loadtxt(filepath_shear, skiprows=16, max_rows=8)
/tmp/ipykernel_1798285/3261894244.py:29: UserWarning: loadtxt: input contained no data: "/renoir/fromenti/Documents/codes_Bulk_flow/results/shear_moment_data_with_Qzz_wMLE.txt"
  cov = np.loadtxt(filepath_shear, skiprows=16, max_rows=8)


In [655]:
# results of the DESI V4 TF file: 
Input_v4   = '/renoir/fromenti/Documents/codes_Bulk_flow/results_CF4_TF_data/'
Bx_eta_v4, By_eta_v4, Bz_eta_v4, B_ampl_eta_v4, err_Bx_eta_v4, err_By_eta_v4,  err_Bz_eta_v4, err_B_ampl_eta_v4, Qxx_eta_v4,  Qxy_eta_v4, Qxz_eta_v4, Qyy_eta_v4, Qyz_eta_v4,Qzz_eta_v4 ,err_Qxx_eta_v4, err_Qxy_eta_v4, err_Qxz_eta_v4, err_Qyy_eta_v4,  err_Qyz_eta_v4, err_Qzz_eta_v4, cov_eta_v4 , Bx_mocks_CRMS_eta_v4, By_mocks_CRMS_eta_v4, Bz_mocks_CRMS_eta_v4, Ux_mocks_CRMS_eta_v4, Uy_mocks_CRMS_eta_v4, Uz_mocks_CRMS_eta_v4, Qxx_opt_mocks_eta_v4, Qxy_opt_mocks_eta_v4, Qxz_opt_mocks_eta_v4, Qyy_opt_mocks_eta_v4, Qyz_opt_mocks_eta_v4, Qzz_opt_mocks_eta_v4,Qxx_true_mocks_eta_v4, Qxy_true_mocks_eta_v4, Qxz_true_mocks_eta_v4, Qyy_true_mocks_eta_v4, Qyz_true_mocks_eta_v4, Qzz_true_mocks_eta_v4 = load_data_results(Input_v4 + 'Bulk_flow_data_etaMLE.txt', Input_v4 + 'shear_moment_data_etaMLE.txt', Input + 'Bulk_flow_all_mocks_etaMLE.txt', Input + 'Shear_moment_all_mocks_with_Qzz_etaMLE_final.txt')
Bx_w_v4, By_w_v4, Bz_w_v4, B_ampl_w_v4, err_Bx_w_v4, err_By_w_v4,  err_Bz_w_v4, err_B_ampl_w_v4, Qxx_w_v4,  Qxy_w_v4, Qxz_w_v4, Qyy_w_v4, Qyz_w_v4,Qzz_w_v4 ,err_Qxx_w_v4, err_Qxy_w_v4, err_Qxz_w_v4, err_Qyy_w_v4,  err_Qyz_w_v4, err_Qzz_w_v4, cov_w_v4 , Bx_mocks_CRMS_w_v4, By_mocks_CRMS_w_v4, Bz_mocks_CRMS_w_v4, Ux_mocks_CRMS_w_v4, Uy_mocks_CRMS_w_v4, Uz_mocks_CRMS_w_v4, Qxx_opt_mocks_w_v4, Qxy_opt_mocks_w_v4, Qxz_opt_mocks_w_v4, Qyy_opt_mocks_w_v4, Qyz_opt_mocks_w_v4, Qzz_opt_mocks_w_v4,Qxx_true_mocks_w_v4, Qxy_true_mocks_w_v4, Qxz_true_mocks_w_v4, Qyy_true_mocks_w_v4, Qyz_true_mocks_w_v4, Qzz_true_mocks_w_v4 = load_data_results( Input_v4 + 'Bulk_flow_data_wMLE.txt', Input_v4 + 'shear_moment_data_wMLE.txt', Input + 'Bulk_flow_all_mocks_wMLE.txt', Input + 'Shear_moment_all_mocks_with_Qzz_wMLE_final.txt')

In [658]:
# chi2 of all the bulk flow and the shear moments:

def chi2_lu(B_measured,B_pq, C_pq):
    
    B_measured_transposée = B_measured.T
    Tot_pq = B_pq + C_pq
    Tot_pq_inv = np.linalg.inv(Tot_pq)
    # ici la matrice B_measured est la matrice des datas
    chi2 = B_measured @ Tot_pq_inv @ B_measured_transposée
    
    return chi2

In [667]:

R_mean_tot = R_mean[:8, :8] # on enlève la compo Qzz size (8,8)

R_tot_eta_v5 = np.array([Bx_eta_v5,By_eta_v5,Bz_eta_v5,Qxx_eta_v5,Qxy_eta_v5,Qxz_eta_v5,Qyy_eta_v5,Qyz_eta_v5]) # le vecteur Up
chi2_tot_eta_v5 = chi2_lu(R_tot_eta_v5,R_mean_tot,cov_eta_v5)  
chi2_norm_tot_eta_v5 = chi2_lu(R_tot_eta_v5,R_mean_tot,cov_eta_v5) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

R_tot_w_v5 = np.array([Bx_w_v5,By_w_v5,Bz_w_v5,Qxx_w_v5,Qxy_w_v5,Qxz_w_v5,Qyy_w_v5,Qyz_w_v5]) 
chi2_tot_w_v5 = chi2_lu(R_tot_w_v5,R_mean_tot,cov_w_v5) 
chi2_norm_tot_w_v5 = chi2_lu(R_tot_w_v5,R_mean_tot,cov_w_v5) / 8  

R_tot_eta_v4 = np.array([Bx_eta_v4,By_eta_v4,Bz_eta_v4,Qxx_eta_v4,Qxy_eta_v4,Qxz_eta_v4,Qyy_eta_v4,Qyz_eta_v4]) # le vecteur Up
chi2_tot_eta_v4 = chi2_lu(R_tot_eta_v4,R_mean_tot,cov_eta_v4)  
chi2_norm_tot_eta_v4 = chi2_lu(R_tot_eta_v4,R_mean_tot,cov_eta_v4) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

R_tot_w_v4 = np.array([Bx_w_v4,By_w_v4,Bz_w_v4,Qxx_w_v4,Qxy_w_v4,Qxz_w_v4,Qyy_w_v4,Qyz_w_v4]) 
chi2_tot_w_v4 = chi2_lu(R_tot_w_v4,R_mean_tot,cov_w_v4) 
chi2_norm_tot_w_v4 = chi2_lu(R_tot_w_v4,R_mean_tot,cov_w_v4) / 8  

R_fei_tot = R_theorie[:8, :8] 
chi2_tot_fei = chi2_lu(R_tot_eta_v5,R_fei_tot,cov_eta_v5) 
chi2_norm_tot_fei = chi2_lu(R_tot_eta_v5,R_fei_tot,cov_eta_v5) / 8 

print('The chi2 normalized of the prediction versus the data v5 for etaMLE is:',chi2_norm_tot_eta_v5)
print('The chi2 normalized of the prediction from fei  versus the data v5 for etaMLE is:',chi2_norm_tot_fei)
print('The chi2 normalized of the prediction versus the data v5 for wMLE is:',chi2_norm_tot_w_v5)
print('The chi2 normalized of the prediction versus the data v4 for etaMLE is:',chi2_norm_tot_eta_v4)
print('The chi2 normalized of the prediction versus the data v4 for wMLE is:',chi2_norm_tot_w_v4)


The chi2 normalized of the prediction versus the data v5 for etaMLE is: 3.326880607593096
The chi2 normalized of the prediction from fei  versus the data v5 for etaMLE is: 3.432812239273092
The chi2 normalized of the prediction versus the data v5 for wMLE is: 3.434677863199004
The chi2 normalized of the prediction versus the data v4 for etaMLE is: 11.78574968956411
The chi2 normalized of the prediction versus the data v4 for wMLE is: 12.471008149480868


In [673]:
# p_values calcul:


def p_value_chi2_manual_even_dof(chi2, k):
    """
    Calcul du p-value pour k pair UNIQUEMENT.
    Formule: P(χ² > x) = e^(-x/2) * Σ_{j=0}^{k/2 - 1} (x/2)^j / j!
    """
    if k % 2 != 0:
        raise ValueError("Cette formule fonctionne uniquement pour k pair")
    
    x = chi2
    sum_term = 0
    for j in range(k // 2):
        sum_term += (x/2)**j / math.factorial(j)
    
    p_value = math.exp(-x/2) * sum_term
    return p_value

In [672]:
print('P_value for etaMLE:',p_value_chi2_manual_even_dof((chi2_tot_eta_v5), 8))  
print('P_value for wMLE:  ',p_value_chi2_manual_even_dof((chi2_tot_w_v5), 8))  
print('P_value for fei   :',p_value_chi2_manual_even_dof((chi2_tot_fei), 8))

dof = 8
p_value_eta = chi2.sf(chi2_tot_eta_v5, dof)  # sf = survival function = 1 - cdf
p_value_w = chi2.sf(chi2_tot_w_v5, dof)
p_value_fei = chi2.sf(chi2_tot_fei, dof)

print(f"\np-value (etaMLE) with scipy = {p_value_eta:.7f}")
print(f"p-value (wMLE) with scipy   = {p_value_w:.7f}")
print(f"p-value (fei) with scipy    = {p_value_fei:.7f}")

P_value for etaMLE: 0.0008237002071165368
P_value for wMLE:   0.0005845274718718642
P_value for fei   : 0.000588020414438734

p-value (etaMLE) with scipy = 0.0008237
p-value (wMLE) with scipy   = 0.0005845
p-value (fei) with scipy    = 0.0005880


In [674]:
def bulk_flow_direction(Bx, By, Bz, err_Bx, err_By, err_Bz):
    
    # Amplitude et son erreur
    B_ampl = np.sqrt(Bx**2 + By**2 + Bz**2)
    err_B_ampl = (np.abs(Bx*err_Bx) + np.abs(By*err_By) + np.abs(Bz*err_Bz)) / B_ampl
    
    # Conversion en coordonnées galactiques
    sph = SkyCoord(w=Bx*u.km/u.s, u=By*u.km/u.s, v=Bz*u.km/u.s, frame='galactic', representation_type='cartesian').spherical
    l, b = sph.lon.deg, sph.lat.deg
    
    # Erreurs sur l et b
    R_perp = np.sqrt(Bx**2 + By**2)
    err_l_deg = np.degrees(np.sqrt(((-By/R_perp**2)*err_Bx)**2 + ((Bx/R_perp**2)*err_By)**2)) if R_perp > 0 else 0
    err_b_deg = np.degrees(np.sqrt(((-Bx*Bz/(B_ampl**2*R_perp))*err_Bx)**2 + ((-By*Bz/(B_ampl**2*R_perp))*err_By)**2 + ((R_perp/B_ampl**2)*err_Bz)**2)) if (B_ampl > 0 and R_perp > 0) else 0
    
    return l, err_l_deg, b, err_b_deg

In [ ]:

l_eta_v5, err_l_eta_v5, b_eta_v5, err_b_eta_v5 = bulk_flow_direction(Bx_eta_v5, By_eta_v5, Bz_eta_v5,err_Bx_eta_v5, err_By_eta_v5, err_Bz_eta_v5)
l_w_v5, err_l_w_v5, b_w_v5, err_b_w_v5 = bulk_flow_direction(Bx_w_v5, By_w_v5, Bz_w_v5, err_Bx_w_v5, err_By_w_v5, err_Bz_w_v5)
l_eta_v4, err_l_eta_v4, b_eta_v4, err_b_eta_v4 = bulk_flow_direction(Bx_eta_v4, By_eta_v4, Bz_eta_v4,err_Bx_eta_v4, err_By_eta_v4, err_Bz_eta_v4)
l_w_v4, err_l_w_v4, b_w_v4, err_b_w_v4 = bulk_flow_direction(Bx_w_v4, By_w_v4, Bz_w_v4, err_Bx_w_v4, err_By_w_v4, err_Bz_w_v4)
print("="*60)
print("Bulk Flow direction etaMLE avec incertitudes data v5")
print("="*60)
print(f"Longitude l = {l_eta_v5:.1f}° ± {err_l_etav5:.1f}°")
print(f"Latitude b = {b_eta_v5:.1f}° ± {err_b_eta_v5:.1f}°")

print("="*60)
print("Bulk Flow direction wMLE avec incertitudes data v5")
print("="*60)
print(f"Longitude l = {l_w_v5:.1f}° ± {err_l_w_v5:.1f}°")
print(f"Latitude b = {b_w_v5:.1f}° ± {err_b_w_v5:.1f}°")
print("="*60)

print("Bulk Flow direction etaMLE avec incertitudes data v4")
print("="*60)
print(f"Longitude l = {l_eta:.1f}° ± {err_l_eta:.1f}°")
print(f"Latitude b = {b_eta:.1f}° ± {err_b_eta:.1f}°")

print("="*60)
print("Bulk Flow direction wMLE avec incertitudes data v4")
print("="*60)
print(f"Longitude l = {l_w:.1f}° ± {err_l_w:.1f}°")
print(f"Latitude b = {b_w:.1f}° ± {err_b_w:.1f}°")

Bulk Flow direction etaMLE avec incertitudes
Longitude l = 269.7° ± 10.0°
Latitude b = -17.8° ± 2.1°
Bulk Flow direction wMLE avec incertitudes
Longitude l = 271.4° ± 10.2°
Latitude b = -17.5° ± 2.2°


In [603]:
print("Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies")
print("="*100)
print("Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_eta:6.2f} ± {err_Bx_eta:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_eta:6.2f}              ± {Bx_mocks_CRMS_eta:6.2f} ")
print(f"By  (km/s)           {By_data_eta:6.2f} ± {err_By_eta:5.2f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_eta:6.2f}              ± {By_mocks_CRMS_eta:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_eta:6.2f} ± {err_Bz_eta:5.2f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_eta:6.2f}              ± {Bz_mocks_CRMS_eta:6.2f} ")
print(f"|B| (km/s)           {B_data_eta:6.2f} ± {err_B_eta:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_eta:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_eta:6.3f} ± {error_Qxx_eta:5.3f}      ± {CRMS_Qxx:6.3f}          ± {Qxx_true_mocks_eta:6.3f}              ± {Qxx_opt_mocks_eta:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_eta:6.3f} ± {error_Qxy_eta:5.3f}      ± {CRMS_Qxy:6.3f}          ± {Qxy_true_mocks_eta:6.3f}              ± {Qxy_opt_mocks_eta:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_eta:6.3f} ± {error_Qxz_eta:5.3f}      ± {CRMS_Qxz:6.3f}          ± {Qxz_true_mocks_eta:6.3f}              ± {Qxz_opt_mocks_eta:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_eta:6.3f} ± {error_Qyy_eta:5.3f}      ± {CRMS_Qyy:6.3f}          ± {Qyy_true_mocks_eta:6.3f}              ± {Qyy_opt_mocks_eta:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_eta:6.3f} ± {error_Qyz_eta:5.3f}      ± {CRMS_Qyz:6.3f}          ± {Qyz_true_mocks_eta:6.3f}              ± {Qyz_opt_mocks_eta:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_eta:6.3f} ± {error_Qzz_eta:5.3f}      ± {CRMS_Qzz:6.3f}          ± {Qzz_true_mocks_eta:6.3f}              ± {Qzz_opt_mocks_eta:6.3f} ")
print("")
print(f"Chi2                    {chi2_tot_eta:6.3f}           ddof 8                                                           " )   
print(f"Chi2 normalized         {chi2_norm_tot_eta:6.3f}        ")
print(f"P-value                 {p_value_eta:6.5f} "  )
print(f"Direction of the Bulk flow        l = {l_eta:6.2f}° ±{err_l_eta:6.2f}°             b = {b_eta:6.2f}° ±{err_b_eta:6.2f}°             ")

Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies
Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -253.47 ± 21.97      ±  73.24          ±  46.50              ±  64.58 
By  (km/s)            32.56 ± 26.91      ±   86.2          ±  52.61              ±  71.94 
Bz  (km/s)          -278.46 ± 36.11      ±  217.8          ±  98.71              ± 106.52 
|B| (km/s)           377.95 ± 43.66      ±  245.4          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)      0.113 ± 0.161      ±  0.372          ±  0.612              ±  0.449 
Qxy (h*km/s/Mpc)     -0.004 ± 0.153      ±  0.192          ±  0.383              ±  0.292
Qxz (h*km/s/Mpc)     -0.780 ± 0.275      ±  0.305          ±  0.207              ±  0.479
Qyy (h*km/s/Mpc)     -0.815 ± 0.258      ±  0.394          ±  0.370              ±  0.476

In [604]:
print("Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies")
print("="*100)
print("Component           Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_w:6.2f} ± {err_Bx_w:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_w:6.2f}              ± {Bx_mocks_CRMS_w:6.2f} ")
print(f"By  (km/s)           {By_data_w:6.2f} ± {err_By_w:5.2f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_w:6.2f}              ± {By_mocks_CRMS_w:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_w:6.2f} ± {err_Bz_w:5.2f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_w:6.2f}              ± {Bz_mocks_CRMS_w:6.2f} ")
print(f"|B| (km/s)           {B_data_w:6.2f} ± {err_B_w:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_w:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_w:6.3f} ± {error_Qxx_w:5.3f}      ± {CRMS_Qxx:6.3f}          ± {Qxx_true_mocks_w:6.3f}              ± {Qxx_opt_mocks_w:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_w:6.3f} ± {error_Qxy_w:5.3f}      ± {CRMS_Qxy:6.3f}          ± {Qxy_true_mocks_w:6.3f}              ± {Qxy_opt_mocks_w:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_w:6.3f} ± {error_Qxz_w:5.3f}      ± {CRMS_Qxz:6.3f}          ± {Qxz_true_mocks_w:6.3f}              ± {Qxz_opt_mocks_w:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_w:6.3f} ± {error_Qyy_w:5.3f}      ± {CRMS_Qyy:6.3f}          ± {Qyy_true_mocks_w:6.3f}              ± {Qyy_opt_mocks_w:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_w:6.3f} ± {error_Qyz_w:5.3f}      ± {CRMS_Qyz:6.3f}          ± {Qyz_true_mocks_w:6.3f}              ± {Qyz_opt_mocks_w:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_w:6.3f} ± {error_Qzz_w:5.3f}      ± {CRMS_Qzz:6.3f}          ± {Qzz_true_mocks_w:6.3f}              ± {Qzz_opt_mocks_w:6.3f} ")
print('')
print(f"Chi2                    {chi2_tot_w:6.3f}           ddof 8                                                           " )  
print(f"Chi2 normalized         {chi2_norm_tot_w:6.3f} "  )
print(f"P-value                 {p_value_w:6.5f} "  )
print(f"Direction of the Bulk flow        l = {l_w:6.2f}° ±{err_l_w:6.2f}°             b = {b_w:6.2f}° ±{err_b_w:6.2f}°             ")

Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies
Component           Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -259.67 ± 22.31      ±  73.24          ±  46.50              ±  66.19 
By  (km/s)            32.65 ± 26.99      ±   86.2          ±  52.61              ±  73.66 
Bz  (km/s)          -284.63 ± 37.25      ±  217.8          ±  98.71              ± 108.83 
|B| (km/s)           386.66 ± 44.68      ±  245.4          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)      0.130 ± 0.178      ±  0.372          ±  0.630              ±  0.463 
Qxy (h*km/s/Mpc)     -0.008 ± 0.154      ±  0.192          ±  0.383              ±  0.301
Qxz (h*km/s/Mpc)     -0.792 ± 0.288      ±  0.305          ±  0.207              ±  0.494
Qyy (h*km/s/Mpc)     -0.836 ± 0.267      ±  0.394          ±  0.370              ±  0.490

In [605]:
print("Table - Bulk Flow and shear moment Measured from DESI DR1 from Fei matrix with (ηMLE)")
print("="*100)
print("Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_eta:6.2f} ± {err_Bx_eta:5.2f}      ± {Bx_CRMS_fei:6.2f}          ± {Ux_mocks_CRMS_eta:6.2f}              ± {Bx_mocks_CRMS_eta:6.2f} ")
print(f"By  (km/s)           {By_data_eta:6.2f} ± {err_By_eta:5.2f}      ± {By_CRMS_fei:6.1f}          ± {Uy_mocks_CRMS_eta:6.2f}              ± {By_mocks_CRMS_eta:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_eta:6.2f} ± {err_Bz_eta:5.2f}      ± {Bz_CRMS_fei:6.1f}          ± {Uz_mocks_CRMS_eta:6.2f}              ± {Bz_mocks_CRMS_eta:6.2f} ")
print(f"|B| (km/s)           {B_data_eta:6.2f} ± {err_B_eta:5.2f}      ± {sigma_B_fei:6.1f}          ± {err_U_mocks_true_eta:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_eta:6.3f} ± {error_Qxx_eta:5.3f}      ± {CRMS_Qxx_fei:6.3f}          ± {Qxx_true_mocks_eta:6.3f}              ± {Qxx_opt_mocks_eta:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_eta:6.3f} ± {error_Qxy_eta:5.3f}      ± {CRMS_Qxy_fei:6.3f}          ± {Qxy_true_mocks_eta:6.3f}              ± {Qxy_opt_mocks_eta:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_eta:6.3f} ± {error_Qxz_eta:5.3f}      ± {CRMS_Qxz_fei:6.3f}          ± {Qxz_true_mocks_eta:6.3f}              ± {Qxz_opt_mocks_eta:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_eta:6.3f} ± {error_Qyy_eta:5.3f}      ± {CRMS_Qyy_fei:6.3f}          ± {Qyy_true_mocks_eta:6.3f}              ± {Qyy_opt_mocks_eta:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_eta:6.3f} ± {error_Qyz_eta:5.3f}      ± {CRMS_Qyz_fei:6.3f}          ± {Qyz_true_mocks_eta:6.3f}              ± {Qyz_opt_mocks_eta:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_eta:6.3f} ± {error_Qzz_eta:5.3f}      ± {CRMS_Qzz_fei:6.3f}          ± {Qzz_true_mocks_eta:6.3f}              ± {Qzz_opt_mocks_eta:6.3f} ")
print('')
print(f"Chi2                    {chi2_tot_eta:6.3f}           ddof 8                                                           " )  
print(f"Chi2 normalized         {chi2_norm_tot_eta:6.3f} "  )
print(f"P-value                 {p_value_eta:6.5f} "  )
print(f"Direction of the Bulk flow        l = {l_eta:6.2f}° ±{err_l_eta:6.2f}°             b = {b_eta:6.2f}° ±{err_b_eta:6.2f}°             ")

Table - Bulk Flow and shear moment Measured from DESI DR1 from Fei matrix with (ηMLE)
Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -253.47 ± 21.97      ±  72.45          ±  46.50              ±  64.58 
By  (km/s)            32.56 ± 26.91      ±   84.0          ±  52.61              ±  71.94 
Bz  (km/s)          -278.46 ± 36.11      ±  213.2          ±  98.71              ± 106.52 
|B| (km/s)           377.95 ± 43.66      ±  240.3          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)      0.113 ± 0.161      ±  0.374          ±  0.612              ±  0.449 
Qxy (h*km/s/Mpc)     -0.004 ± 0.153      ±  0.190          ±  0.383              ±  0.292
Qxz (h*km/s/Mpc)     -0.780 ± 0.275      ±  0.292          ±  0.207              ±  0.479
Qyy (h*km/s/Mpc)     -0.815 ± 0.258      ±  0.390          ±  0.370             

## CF4 TF data catalog :

In [606]:
# B = np.sqrt( Bx**2 + By**2 + Bz**2)
# pour l'estimateur etaMLE:
# We import the bulk flow results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results_CF4_TF_data/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'
Fit_tech2 = 'wMLE'  

# We load the bulk flow results of the data:
Bx_data_eta, By_data_eta, Bz_data_eta = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_eta, err_By_eta ,err_Bz_eta = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech1}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_eta = np.sqrt( Bx_data_eta**2 + By_data_eta**2 + Bz_data_eta**2)
# et on fait la propagation des incertitudes sur B:
err_B_eta = (np.abs(Bx_data_eta*err_Bx_eta) + np.abs(By_data_eta*err_By_eta) + np.abs(Bz_data_eta*err_Bz_eta)) / B_data_eta

# We load the bulk flow results of the data:
Bx_data_w, By_data_w, Bz_data_w = np.loadtxt(Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt',skiprows=6,usecols=1,max_rows=3)
# we load the error on the Bx,By,Bz estimation for the data:
err_Bx_w, err_By_w ,err_Bz_w = np.loadtxt( Input_dir + f'Bulk_flow_data_{Fit_tech2}.txt', skiprows=6, usecols=2, max_rows=3)

B_data_w = np.sqrt( Bx_data_w**2 + By_data_w**2 + Bz_data_w**2)
# et on fait la propagation des incertitudes sur B:
err_B_w = (np.abs(Bx_data_w*err_Bx_w) + np.abs(By_data_w*err_By_w) + np.abs(Bz_data_w*err_Bz_w)) / B_data_w

print('For estimator etaMLE the bulk flow is:',B_data_eta,'+-',err_B_eta,'km/s')
print('For estimator wMLE the bulk flow is:',B_data_w,'+-',err_B_w,'km/s')
print(' For estimator eta: ')
print('Bx=',Bx_data_eta,'+-',err_Bx_eta,'km/s')
print('By=',By_data_eta,'+-',err_By_eta,'km/s')
print('Bz=',Bz_data_eta,'+-',err_Bz_eta,'km/s')
print(' For estimator wMLE:')
print('Bx=',Bx_data_w,'+-',err_Bx_w,'km/s')
print('By=',By_data_w,'+-',err_By_w,'km/s')
print('Bz=',Bz_data_w,'+-',err_Bz_w,'km/s')

For estimator etaMLE the bulk flow is: 334.72027758080213 +- 19.11731623381458 km/s
For estimator wMLE the bulk flow is: 334.07290809847086 +- 20.099823269316076 km/s
 For estimator eta: 
Bx= -102.59594649340596 +- 11.830134661088689 km/s
By= -1.9006756315064142 +- 17.947029471566253 km/s
Bz= -318.60339517813554 +- 16.16780652707406 km/s
 For estimator wMLE:
Bx= -100.53354347063748 +- 12.22270108868303 km/s
By= 7.623599878359844 +- 17.949052615206373 km/s
Bz= -318.49583244918836 +- 16.795128916286334 km/s


In [607]:
# pour l'estimateur etaMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results_CF4_TF_data/'
Fit_tech1 = 'etaMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_eta = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech1}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_eta = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech1}.txt', skiprows=13, usecols=2, max_rows=1)


print('For estimator etaMLE the Qxx component of the shear moment is:',Qxx_data_eta,'+-',error_Qxx_eta,'km/s')
print('For estimator etaMLE the Qyy component of the shear moment is:',Qyy_data_eta,'+-',error_Qyy_eta,'km/s')
print('For estimator etaMLE the Qzz component of the shear moment is:',Qzz_data_eta,'+-',error_Qzz_eta,'km/s')
print('For estimator etaMLE the Qxy component of the shear moment is:',Qxy_data_eta,'+-',error_Qxy_eta,'km/s')
print('For estimator etaMLE the Qxz component of the shear moment is:',Qxz_data_eta,'+-',error_Qxz_eta,'km/s')
print('For estimator etaMLE the Qyz component of the shear moment is:',Qyz_data_eta,'+-',error_Qyz_eta,'km/s')

For estimator etaMLE the Qxx component of the shear moment is: -1.1135 +- 0.415 km/s
For estimator etaMLE the Qyy component of the shear moment is: 0.1323 +- 0.6424 km/s
For estimator etaMLE the Qzz component of the shear moment is: 0.9812 +- 0.7648 km/s
For estimator etaMLE the Qxy component of the shear moment is: 2.3978 +- 0.3889 km/s
For estimator etaMLE the Qxz component of the shear moment is: 3.5678 +- 0.4356 km/s
For estimator etaMLE the Qyz component of the shear moment is: 2.6365 +- 0.5981 km/s


In [608]:
# pour l'estimateur wMLE:
# We import the shear moment results from the .txt files
Input_dir   = '/renoir/fromenti/Documents/codes_Bulk_flow/results_CF4_TF_data/'
Fit_tech2 = 'wMLE'  # Fit_tech can be 'etaMLE', 'wMLE' or 'tMLE'

# We load the bulk flow results of the data:
Qxx_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=8,usecols=1,max_rows=1)
Qxy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=9,usecols=1,max_rows=1)
Qxz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=10,usecols=1,max_rows=1)
Qyy_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=11,usecols=1,max_rows=1)
Qyz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=12,usecols=1,max_rows=1)
Qzz_data_w = np.loadtxt(Input_dir + f'shear_moment_data_{Fit_tech2}.txt',skiprows=13,usecols=1,max_rows=1)

# we load the error on the Bx,By,Bz estimation for the data:
error_Qxx_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=8, usecols=2, max_rows=1)
error_Qxy_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=9, usecols=2, max_rows=1)
error_Qxz_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=10, usecols=2, max_rows=1)
error_Qyy_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=11, usecols=2, max_rows=1)
error_Qyz_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=12, usecols=2, max_rows=1)
error_Qzz_w = np.loadtxt( Input_dir + f'shear_moment_data_{Fit_tech2}.txt', skiprows=13, usecols=2, max_rows=1)


print('For estimator wMLE the Qxx component of the shear moment is:',Qxx_data_w,'+-',error_Qxx_w,'km/s')
print('For estimator wMLE the Qyy component of the shear moment is:',Qyy_data_w,'+-',error_Qyy_w,'km/s')
print('For estimator wMLE the Qzz component of the shear moment is:',Qzz_data_w,'+-',error_Qzz_w,'km/s')
print('For estimator wMLE the Qxy component of the shear moment is:',Qxy_data_w,'+-',error_Qxy_w,'km/s')
print('For estimator wMLE the Qxz component of the shear moment is:',Qxz_data_w,'+-',error_Qxz_w,'km/s')
print('For estimator wMLE the Qyz component of the shear moment is:',Qyz_data_w,'+-',error_Qyz_w,'km/s')

For estimator wMLE the Qxx component of the shear moment is: -1.0961 +- 0.3937 km/s
For estimator wMLE the Qyy component of the shear moment is: -0.0477 +- 0.6405 km/s
For estimator wMLE the Qzz component of the shear moment is: 1.1438 +- 0.7518 km/s
For estimator wMLE the Qxy component of the shear moment is: 2.4497 +- 0.3787 km/s
For estimator wMLE the Qxz component of the shear moment is: 3.6314 +- 0.4452 km/s
For estimator wMLE the Qyz component of the shear moment is: 2.5836 +- 0.6044 km/s


In [609]:
# compute the chi2:
R_tot_eta = np.array([Bx_data_eta,By_data_eta,Bz_data_eta,Qxx_data_eta,Qxy_data_eta,Qxz_data_eta,Qyy_data_eta,Qyz_data_eta]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_tot_eta = chi2_lu(R_tot_eta,R_mean_tot,C_pq_etaMLE_tot)  
chi2_norm_tot_eta = chi2_lu(R_tot_eta,R_mean_tot,C_pq_etaMLE_tot) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

R_tot_w = np.array([Bx_data_w,By_data_w,Bz_data_w,Qxx_data_w,Qxy_data_w,Qxz_data_w,Qyy_data_w,Qyz_data_w]) # la matrice B qui est comme Up mais ici on prend que le bulk flow par simplicité
chi2_tot_w = chi2_lu(R_tot_w,R_mean_tot,C_pq_wMLE_tot) 
chi2_norm_tot_w = chi2_lu(R_tot_w,R_mean_tot,C_pq_wMLE_tot) / 8  # car 9 composante - une contrainte de trace nulle ou 8 si on enleve juste Qzz

print('The chi2 normalized of the prediction versus the data for etaMLE is:',chi2_tot_eta)
print('The chi2 normalized of the prediction versus the data for wMLE is:',chi2_tot_w)

# p_values:
dof = 8
p_value_eta = chi2.sf(chi2_tot_eta, dof)  # sf = survival function = 1 - cdf
p_value_w = chi2.sf(chi2_tot_w, dof)

print(f"\np-value (etaMLE) with scipy = {p_value_eta}")
print(f"p-value (wMLE) with scipy = {p_value_w}")


The chi2 normalized of the prediction versus the data for etaMLE is: 225.1081804276704
The chi2 normalized of the prediction versus the data for wMLE is: 214.5982355565738

p-value (etaMLE) with scipy = 3.2058147539102477e-44
p-value (wMLE) with scipy = 5.32623711225675e-42


In [610]:
# THe direction of the bulk flow:

l_eta, err_l_eta, b_eta, err_b_eta, B_amp_eta, err_B_amp_eta = bulk_flow_with_errors(Bx_data_eta, By_data_eta, Bz_data_eta,err_Bx_eta, err_By_eta, err_Bz_eta)
l_w, err_l_w, b_w, err_b_w, B_amp_w, err_B_amp_w = bulk_flow_with_errors(Bx_data_w, By_data_w, Bz_data_w, err_Bx_w, err_By_w, err_Bz_w)

print(f"Longitude l = {l_eta:.1f}° ± {err_l_eta:.1f}°")
print(f"Latitude b = {b_eta:.1f}° ± {err_b_eta:.1f}°")


print(f"Longitude l = {l_w:.1f}° ± {err_l_w:.1f}°")
print(f"Latitude b = {b_w:.1f}° ± {err_b_w:.1f}°")

Longitude l = 269.7° ± 10.0°
Latitude b = -17.8° ± 2.1°
Longitude l = 271.4° ± 10.2°
Latitude b = -17.5° ± 2.2°


In [611]:
print("Table - Bulk Flow and shear moment Measured from CF4 TF Data")
print("="*100)
print("Component           Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_w:6.2f} ± {err_Bx_w:5.2f}      ± {Bx_CRMS:6.2f}          ± {Ux_mocks_CRMS_w:6.2f}              ± {Bx_mocks_CRMS_w:6.2f} ")
print(f"By  (km/s)           {By_data_w:6.2f} ± {err_By_w:5.2f}      ± {By_CRMS:6.1f}          ± {Uy_mocks_CRMS_w:6.2f}              ± {By_mocks_CRMS_w:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_w:6.2f} ± {err_Bz_w:5.2f}      ± {Bz_CRMS:6.1f}          ± {Uz_mocks_CRMS_w:6.2f}              ± {Bz_mocks_CRMS_w:6.2f} ")
print(f"|B| (km/s)           {B_data_w:6.2f} ± {err_B_w:5.2f}      ± {sigma_B:6.1f}          ± {err_U_mocks_true_w:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_w:6.3f} ± {error_Qxx_w:5.3f}      ± {CRMS_Qxx:6.3f}          ± {Qxx_true_mocks_w:6.3f}              ± {Qxx_opt_mocks_w:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_w:6.3f} ± {error_Qxy_w:5.3f}      ± {CRMS_Qxy:6.3f}          ± {Qxy_true_mocks_w:6.3f}              ± {Qxy_opt_mocks_w:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_w:6.3f} ± {error_Qxz_w:5.3f}      ± {CRMS_Qxz:6.3f}          ± {Qxz_true_mocks_w:6.3f}              ± {Qxz_opt_mocks_w:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_w:6.3f} ± {error_Qyy_w:5.3f}      ± {CRMS_Qyy:6.3f}          ± {Qyy_true_mocks_w:6.3f}              ± {Qyy_opt_mocks_w:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_w:6.3f} ± {error_Qyz_w:5.3f}      ± {CRMS_Qyz:6.3f}          ± {Qyz_true_mocks_w:6.3f}              ± {Qyz_opt_mocks_w:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_w:6.3f} ± {error_Qzz_w:5.3f}      ± {CRMS_Qzz:6.3f}          ± {Qzz_true_mocks_w:6.3f}              ± {Qzz_opt_mocks_w:6.3f} ")
print('')
print(f"Chi2                    {chi2_tot_w:6.3f}           ddof 8                                                           " )  
print(f"Chi2 normalized         {chi2_norm_tot_w:6.3f} "  )
print(f"P-value                 {p_value_w} "  )
print(f"Direction of the Bulk flow        l = {l_w:6.2f}° ±{err_l_w:6.2f}°             b = {b_w:6.2f}° ±{err_b_w:6.2f}°             ")

Table - Bulk Flow and shear moment Measured from CF4 TF Data
Component           Measured (wMLE)     CRMS (ΛCDM)     True mocks(wMLE)    Estimated mocks(wMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -100.53 ± 12.22      ±  73.24          ±  46.50              ±  66.19 
By  (km/s)             7.62 ± 17.95      ±   86.2          ±  52.61              ±  73.66 
Bz  (km/s)          -318.50 ± 16.80      ±  217.8          ±  98.71              ± 108.83 
|B| (km/s)           334.07 ± 20.10      ±  245.4          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)     -1.096 ± 0.394      ±  0.372          ±  0.630              ±  0.463 
Qxy (h*km/s/Mpc)      2.450 ± 0.379      ±  0.192          ±  0.383              ±  0.301
Qxz (h*km/s/Mpc)      3.631 ± 0.445      ±  0.305          ±  0.207              ±  0.494
Qyy (h*km/s/Mpc)     -0.048 ± 0.640      ±  0.394          ±  0.370              ±  0.490 
Qyz (h*km/s/Mp

In [612]:
print("Table - Bulk Flow and shear moment Measured from FCF4 TF Data")
print("="*100)
print("Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)")
print("-"*100)
print(f"Bx  (km/s)          {Bx_data_eta:6.2f} ± {err_Bx_eta:5.2f}      ± {Bx_CRMS_fei:6.2f}          ± {Ux_mocks_CRMS_eta:6.2f}              ± {Bx_mocks_CRMS_eta:6.2f} ")
print(f"By  (km/s)           {By_data_eta:6.2f} ± {err_By_eta:5.2f}      ± {By_CRMS_fei:6.1f}          ± {Uy_mocks_CRMS_eta:6.2f}              ± {By_mocks_CRMS_eta:6.2f} ")
print(f"Bz  (km/s)          {Bz_data_eta:6.2f} ± {err_Bz_eta:5.2f}      ± {Bz_CRMS_fei:6.1f}          ± {Uz_mocks_CRMS_eta:6.2f}              ± {Bz_mocks_CRMS_eta:6.2f} ")
print(f"|B| (km/s)           {B_data_eta:6.2f} ± {err_B_eta:5.2f}      ± {sigma_B_fei:6.1f}          ± {err_U_mocks_true_eta:6.2f}              ± {err_B_mocks_opt_eta:6.2f} ")
print(f"Qxx (h*km/s/Mpc)     {Qxx_data_eta:6.3f} ± {error_Qxx_eta:5.3f}      ± {CRMS_Qxx_fei:6.3f}          ± {Qxx_true_mocks_eta:6.3f}              ± {Qxx_opt_mocks_eta:6.3f} ")
print(f"Qxy (h*km/s/Mpc)     {Qxy_data_eta:6.3f} ± {error_Qxy_eta:5.3f}      ± {CRMS_Qxy_fei:6.3f}          ± {Qxy_true_mocks_eta:6.3f}              ± {Qxy_opt_mocks_eta:6.3f}")
print(f"Qxz (h*km/s/Mpc)     {Qxz_data_eta:6.3f} ± {error_Qxz_eta:5.3f}      ± {CRMS_Qxz_fei:6.3f}          ± {Qxz_true_mocks_eta:6.3f}              ± {Qxz_opt_mocks_eta:6.3f}")
print(f"Qyy (h*km/s/Mpc)     {Qyy_data_eta:6.3f} ± {error_Qyy_eta:5.3f}      ± {CRMS_Qyy_fei:6.3f}          ± {Qyy_true_mocks_eta:6.3f}              ± {Qyy_opt_mocks_eta:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz_data_eta:6.3f} ± {error_Qyz_eta:5.3f}      ± {CRMS_Qyz_fei:6.3f}          ± {Qyz_true_mocks_eta:6.3f}              ± {Qyz_opt_mocks_eta:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz_data_eta:6.3f} ± {error_Qzz_eta:5.3f}      ± {CRMS_Qzz_fei:6.3f}          ± {Qzz_true_mocks_eta:6.3f}              ± {Qzz_opt_mocks_eta:6.3f} ")
print('')
print(f"Chi2                    {chi2_tot_eta:6.3f}           ddof 8                                                           " )  
print(f"Chi2 normalized         {chi2_norm_tot_eta:6.3f} "  )
print(f"P-value                 {p_value_eta} "  )
print(f"Direction of the Bulk flow        l = {l_eta:6.2f}° ±{err_l_eta:6.2f}°             b = {b_eta:6.2f}° ±{err_b_eta:6.2f}°             ")

Table - Bulk Flow and shear moment Measured from FCF4 TF Data
Component           Measured (ηMLE)     CRMS (ΛCDM)     True mocks(ηMLE)    Estimated mocks(ηMLE)
----------------------------------------------------------------------------------------------------
Bx  (km/s)          -102.60 ± 11.83      ±  72.45          ±  46.50              ±  64.58 
By  (km/s)            -1.90 ± 17.95      ±   84.0          ±  52.61              ±  71.94 
Bz  (km/s)          -318.60 ± 16.17      ±  213.2          ±  98.71              ± 106.52 
|B| (km/s)           334.72 ± 19.12      ±  240.3          ±  58.19              ±  62.10 
Qxx (h*km/s/Mpc)     -1.113 ± 0.415      ±  0.374          ±  0.612              ±  0.449 
Qxy (h*km/s/Mpc)      2.398 ± 0.389      ±  0.190          ±  0.383              ±  0.292
Qxz (h*km/s/Mpc)      3.568 ± 0.436      ±  0.292          ±  0.207              ±  0.479
Qyy (h*km/s/Mpc)      0.132 ± 0.642      ±  0.390          ±  0.370              ±  0.476 
Qyz (h*km/s/M